In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score

In [2]:
data_file_path = "data/dailog_acts.dat"

In [3]:
data = []
with open(data_file_path, "r") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue
        
        act, utterance = line.split(maxsplit=1)
        #lowercasing
        data.append({"act": str.lower(act), "utterance": str.lower(utterance)})

In [4]:
df = pd.DataFrame(data=data)

In [5]:
df[df["act"] == "inform"]

,act,utterance
0,inform,im looking for a moderately priced restaurant ...
1,inform,any part of town
2,inform,bistro food
6,inform,moderately priced restaurant in the south part...
7,inform,any
...,...,...
23986,inform,thai food
23987,inform,thai food
23988,inform,tailand
23989,inform,thai food in the west of town


In [6]:
df.sample(15)

,act,utterance
6506,inform,i dont care
14969,inform,chinese food east part of town
3901,request,phone number
23321,request,and what type of food
18716,null,noise
1841,request,okay what is their address
21188,null,noise
18058,request,address
6052,request,whats the phone number
23080,null,would you like something


In [7]:
df["act"].unique()

<StringArray>
[  'inform',   'affirm',  'request', 'thankyou',     'null',      'bye',
  'reqalts',   'negate',  'confirm',    'hello',   'repeat',      'ack',
     'deny',  'restart',  'reqmore']
Length: 15, dtype: str

In [8]:
train_data, test_data = train_test_split(df,test_size=0.15,random_state=42)

In [9]:
word_counts = {'inform' : {},   'affirm': {},  'request': {}, 'thankyou' : {},     'null': {},      'bye': {},
  'reqalts': {},   'negate': {},  'confirm': {},    'hello' : {},   'repeat': {},      'ack': {},
     'deny':{},  'restart':{},  'reqmore':{} }

for _,line in train_data.iterrows():
    act = line["act"]
    words = line["utterance"].split()
    for word in words:
        if word in word_counts[act]:
            word_counts[act][word] += 1
        else:
            word_counts[act][word] = 1

In [10]:
rules = {}

for act, counts in word_counts.items():
    rules[act] = sorted(
        counts,
        key=counts.get,
        reverse=True
)[0]

In [11]:
rules

{'inform': 'food',
 'affirm': 'yes',
 'request': 'the',
 'thankyou': 'thank',
 'null': 'noise',
 'bye': 'good',
 'reqalts': 'about',
 'negate': 'no',
 'confirm': 'it',
 'hello': 'hello',
 'repeat': 'repeat',
 'ack': 'okay',
 'deny': 'wrong',
 'restart': 'start',
 'reqmore': 'more'}

In [12]:
def classify(rules, test_data):
    df = test_data.copy()
    df["pred"] = None

    for index, row in df.iterrows():
        words = row["utterance"].split()

        for act, keyword in rules.items():
            if keyword in words:
                df.loc[index, "pred"] = act
                break
            else:
                df.loc[index, "pred"] = "null"

    return df

In [13]:
pred_df = classify(rules=rules,test_data=test_data)

In [14]:
pred_df.sample(15)

,act,utterance,pred
18853,request,phone number,null
19515,null,unintelligible,null
11790,inform,looking for a restaurant in the north part of ...,request
22321,request,phone number,null
20492,inform,dont care,null
7676,reqalts,how about asian oriental,reqalts
290,request,address,null
19161,inform,dont care,null
17959,affirm,yes,affirm
911,inform,west,null


In [15]:
def evaluate(df):
    y_true = df["act"]
    y_pred = df["pred"]
    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_accuracy:.4f}")

In [16]:
evaluate(pred_df)

Accuracy: 0.5008
Balanced Accuracy: 0.4506
